In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt


In [2]:
data = np.load('./QNN/QNN_results.npz', allow_pickle=True)['results'].item()
len(data)

29001

In [3]:
all_amino_acids = pd.read_csv('./QNN/amino_acids_infos.csv')['name'].tolist()
# all_amino_acids

In [4]:
# data['SMPDTHGRMSMKHLXG'][:, :, 1:]

nonsymmetry

In [5]:
result = {(k1,k2): [] for k1 in all_amino_acids for k2 in all_amino_acids}

for sequence, values in data.items():
    values = np.mean(values, axis=0)
    for i, aa1 in enumerate(sequence):
        for j, aa2 in enumerate(sequence):
            result[(aa1, aa2)].append(values[i, j])

for key, value in result.items():
    if len(value) == 0:
        result[key] = 0

result = {k: np.mean(v) for k, v in result.items()}
df = {
    'source': [],
    'target': [],
    'value': [],
}
for k, v in result.items():
    df['source'].append(k[0])
    df['target'].append(k[1])
    df['value'].append(v)
df = pd.DataFrame(df)
# df = df.pivot(index='source', columns='target', values='value')
df.to_csv('nonsymmetry_result.csv', index=False)

In [6]:
result = {(k1,k2): [] for k1 in all_amino_acids for k2 in all_amino_acids}

for sequence, values in data.items():
    values = np.mean(values, axis=0)
    for i, aa1 in enumerate(sequence):
        for j, aa2 in enumerate(sequence):
            result[(aa1, aa2)].append(values[i, j])

# for key, value in result.items():
#     if len(value) == 0:
#         result[key] = 0

# result = {k: np.mean(v) for k, v in result.items()}
df = {
    'source': [],
    'target': [],
    'value': [],
}
for k, v in result.items():
    for i in v:
        df['source'].append(k[0])
        df['target'].append(k[1])
        df['value'].append(round(i, 3))
df = pd.DataFrame(df)
# df = df.pivot(index='source', columns='target', values='value')
df.to_csv('nonsymmetry_list_result.csv', index=False)

symmetry

In [7]:
result = {(k1,k2): [] for k1 in all_amino_acids for k2 in all_amino_acids}

for sequence, values in data.items():
    values = np.mean(values, axis=0)
    for i, aa1 in enumerate(sequence):
        for j, aa2 in enumerate(sequence):
            result[(aa1, aa2)].append(values[i, j].item())

for i1, key1 in enumerate(all_amino_acids):
    for i2 in range(i1+1, len(all_amino_acids)):
        key2 = all_amino_acids[i2]
        result[(key1, key2)] += result[(key2, key1)]

        if (result[(key1, key2)])==0:
            result[(key1, key2)] = [0]

for i1, key1 in enumerate(all_amino_acids):
    for i2 in range(i1, len(all_amino_acids)):
        key2 = all_amino_acids[i2]
        result[(key1, key2)] = np.mean(result[(key1, key2)])

df = {
    'source': [],
    'target': [],
    'value': [],
}
for i1, key1 in enumerate(all_amino_acids):
    for i2 in range(i1, len(all_amino_acids)):
        key2 = all_amino_acids[i2]
        df['source'].append(key1)
        df['target'].append(key2)
        df['value'].append(result[(key1, key2)])
df = pd.DataFrame(df)
# df = df.pivot(index='source', columns='target', values='value')
df.to_csv('symmetry_result.csv', index=False)

In [4]:
# symmetry and total

with_order_dict = {}
result = {}
for i1, k1 in enumerate(all_amino_acids):
    for i2 in range(i1, len(all_amino_acids)):
        k2 = all_amino_acids[i2]
        result[(k1, k2)] = []

        with_order_dict[(k1, k2)] = (k1, k2)
        with_order_dict[(k2, k1)] = (k1, k2)

for sequence, values in data.items():
    values = np.mean(values, axis=0)
    for i, aa1 in enumerate(sequence):
        for hop in range(1, 9):
            j = i + hop
            if j >= len(sequence):
                continue
            else:
                aa2 = sequence[j]
                result[with_order_dict[(aa1, aa2)]].append(values[i, j].item())
                result[with_order_dict[(aa1, aa2)]].append(values[j, i].item())

for key, value in result.items():
    if len(value) == 0:
        result[key] = 0
    result[key] = np.mean(value)

df = {
    'source': [],
    'target': [],
    'value': [],
}
for key, value in result.items():
    df['source'].append(key[0])
    df['target'].append(key[1])
    df['value'].append(result[key])
df = pd.DataFrame(df)
df['norm_value'] = (df['value'] - df['value'].min()) / (df['value'].max() - df['value'].min())
df.to_csv('symmetry_total_result.csv', index=False)


In [4]:
# symmetry and difference hop

def run(hop):
    with_order_dict = {}
    result = {}
    for i1, k1 in enumerate(all_amino_acids):
        for i2 in range(i1, len(all_amino_acids)):
            k2 = all_amino_acids[i2]
            result[(k1, k2)] = []

            with_order_dict[(k1, k2)] = (k1, k2)
            with_order_dict[(k2, k1)] = (k1, k2)

    for sequence, values in data.items():
        values = np.mean(values, axis=0)
        for i, aa1 in enumerate(sequence):
            j = i + hop
            if j >= len(sequence):
                continue
            else:
                aa2 = sequence[j]
                result[with_order_dict[(aa1, aa2)]].append(values[i, j].item())
                result[with_order_dict[(aa1, aa2)]].append(values[j, i].item())

    for key, value in result.items():
        if len(value) == 0:
            result[key] = 0
        result[key] = np.mean(value)

    df = {
        'source': [],
        'target': [],
        'value': [],
    }
    for key, value in result.items():
        df['source'].append(key[0])
        df['target'].append(key[1])
        df['value'].append(result[key])
    df = pd.DataFrame(df)
    df['norm_value'] = (df['value'] - df['value'].min()) / (df['value'].max() - df['value'].min())
    # df = df.pivot(index='source', columns='target', values='value')
    df.to_csv(f'symmetry_hop{hop}_result.csv', index=False)

for i in range(1, 9):
    run(i)
